<a href="https://colab.research.google.com/github/tomsBifx25/Mus-Glioblastoma-snRNAseq/blob/main/notebooks/GBM_snRNAseq_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Age-Dependent Remodeling of the Glioblastoma Microenvironment Revealed by Single-Nucleus Transcriptomics
---
*Thomas Walsh | March 18, 2026*

Hood College
*   Machine Learning for Bioinformatics (*BIFX 546*), Spring 2026
*   Instructor: Dr. Sarangan (Ravi) Ravichandran
---
## Dataset

Name: longevity-db/mouse-glioblastoma-snRNAseq

Source: Hugging Face
  * https://huggingface.co/datasets/longevity-db/mouse-glioblastoma-snRNAseq | CELLxGENE | GEO: GSE186252

Original Data Source: National Center for Biotechnology Information (NCBI) and Gene Expression Omnibus (GEO)
  * https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE215239

---
## Project Goals

### Research Question:
How does the age-dependent Tumor microenvironment diverge in glioblastoma contexts between young and aged mice?

### Relevance:
Aging is the primary risk factor for glioblastoma (GBM) and significantly correlates with poorer prognosis. While much research focuses on the tumor's genetic mutations, the age-dependent microenvironment remains under-explored. This dataset is unique because it provides high-resolution, single-nucleus RNA sequencing (snRNA-seq) across *"Young"* and *"Old"* cohorts, allowing for a granular look at how the aging brain environment modulates cancer progression and immune evasion.

---
# 01 Setup
First, install any core dependent libraries into the colab environment. For the full list of depenencies, see the `requirements.txt` from the GitHub repository.

## Library Installations:

In [ ]:
!pip install scanpy anndata leidenalg gseapy --quiet

## Importing and loading toolkit libraries
Similarly, for the full list of toolkits used, see the `requirements.txt` from the GitHub repository.

In [ ]:
import scanpy as sc
import numpy as np
import os
import shutil
import subprocess
import matplotlib.pyplot as plt
import gseapy as gp
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

## Data Loading
Data copied from the `data/` section of the GitHub repository, originally downloaded from the NCBI GEO asscession page for: `GSE215239`.

Link for direct download if needed:
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE215239

In [ ]:
print("Starting efficient data download using git sparse-checkout...")

github_repo_url = "https://github.com/tomsBifx25/Mus-Glioblastoma-snRNAseq.git"
temp_sparse_clone_dir = "/content/repo_sparse_temp"
output_dir = "/content/"

# Clean up any previous temporary directory
if os.path.exists(temp_sparse_clone_dir):
    shutil.rmtree(temp_sparse_clone_dir)

# 1. Create a temporary directory and initialize git
print(f"Initializing git repository in {temp_sparse_clone_dir}...")
subprocess.run(["mkdir", "-p", temp_sparse_clone_dir], check=True)
subprocess.run(["git", "init"], cwd=temp_sparse_clone_dir, check=True)

# 2. Configure sparse-checkout to only get the 'data' subdirectory
print("Configuring sparse-checkout for 'data' subdirectory...")
subprocess.run(["git", "sparse-checkout", "init", "--cone"], cwd=temp_sparse_clone_dir, check=True)
subprocess.run(["git", "sparse-checkout", "set", "data"], cwd=temp_sparse_clone_dir, check=True)

# 3. Add remote and pull
print("Adding remote and pulling 'main' branch...")
subprocess.run(["git", "remote", "add", "origin", github_repo_url], cwd=temp_sparse_clone_dir, check=True)
subprocess.run(["git", "pull", "origin", "main"], cwd=temp_sparse_clone_dir, check=True)

# Define the path to the sparse-cloned 'data' directory
data_sub_dir = os.path.join(temp_sparse_clone_dir, 'data')

print(f"Contents of {data_sub_dir} after sparse clone: {os.listdir(data_sub_dir)}")
print("Copying files from sparse-cloned 'data' directory to /content/...")

# Copy all files directly from data_sub_dir to output_dir
for filename in os.listdir(data_sub_dir):
    src_path = os.path.join(data_sub_dir, filename)
    dst_path = os.path.join(output_dir, filename)
    if os.path.isfile(src_path): # Only copy files, not subdirectories
        shutil.copy(src_path, dst_path)
        print(f"Copied {filename} to {output_dir}")

print("All specified files from 'data' subdirectory have been processed.")

# Clean up the temporary sparse-cloned repository directory
if os.path.exists(temp_sparse_clone_dir):
    shutil.rmtree(temp_sparse_clone_dir)
    print(f"Removed temporary sparse-cloned directory: {temp_sparse_clone_dir}")

## Colab Environment
Make directories within the colab environment for ease of access and organization.

In [ ]:
# Define the folder names
folder_old = "Old_cohort"
folder_young = "Young_cohort"

# Create the old folder if it doesn't already exist
os.makedirs(folder_old, exist_ok=True)
print(f"Folder '{folder_old}' created successfully (or already exists).")

# Create the young folder if it doesn't already exist
os.makedirs(folder_young, exist_ok=True)
print(f"Folder '{folder_young}' created successfully (or already exists).")

## Organization of data files into respective folders:

After coping the data file from GitHub, there should be six files in the local environment:
* Old_GBM_barcodes.tsv.gz
* Old_GBM_features.tsv.gz
* Old_GBM_matrix.mtx.gz
* Young_GBM_barcodes.tsv.gz
* Young_GBM_features.tsv.gz
* Young_GBM_matrix.mtx.gz

We want to have them organized into our two cohort groups. Move the *"Old"* and *"Young"* tagged files into each respective cohort folder:
* `/content/Old_cohort/`
* `/content/Young_cohort/`

In [ ]:
source_directory = "/content/"
destination_old = "/content/Old_cohort/"
destination_young = "/content/Young_cohort/"

# Get a list of all files in the source directory
files_in_directory = os.listdir(source_directory)

# Iterate through the files and move where needed
for filename in files_in_directory:
    if filename.startswith("Young_"):
        source_path = os.path.join(source_directory, filename)
        destination_path = os.path.join(destination_young, filename)
        try:
            shutil.move(source_path, destination_path)
            print(f"Moved '{filename}' to '{destination_young}'")
        except Exception as e:
            print(f"Error moving '{filename}': {e}")

    elif filename.startswith("Old_"):
        source_path = os.path.join(source_directory, filename)
        destination_path = os.path.join(destination_old, filename)
        try:
            shutil.move(source_path, destination_path)
            print(f"Moved '{filename}': {e}")
        except Exception as e:
            print(f"Error moving '{filename}': {e}")

---
# 02 Data Cleaning
Creating `adata` data frames for our young and old cohorts. For this analysis, convert to `AnnData` (`.h5ad`) data structures. They are Python-based data structures designed for storing annotated data matrices, commonly used in single-cell analysis.

In [ ]:
adata_young = sc.read_10x_mtx("/content/Young_cohort/",
                              var_names="gene_symbols",
                              cache=True,
                              prefix="Young_GBM_"
)

adata_old = sc.read_10x_mtx("/content/Old_cohort/",
                            var_names="gene_symbols",
                            cache=True,
                            prefix="Old_GBM_"
)

In [ ]:
# adata structures
print(str(adata_young))
print(str(adata_old))

Structure of `adata_young` and `adata_old`.

You can see that `adata_young` contains 12032 cells and 32285 genes, while `adata_old` contains 11330 cells and 32285 genes. Both objects have similar observation metadata, including `n_genes_by_counts` and `total_counts`, and variable metadata like `gene_ids` and `feature_types`. This confirms that both datasets have been loaded and are ready for further quality control and analysis steps.

## Quality Control

In [ ]:
# Before calculating qc_metrics, annotate mitochondrial genes
adata_young.var['mt'] = adata_young.var_names.str.startswith('mt-')
adata_old.var['mt'] = adata_old.var_names.str.startswith('mt-')

# Calculate quality control metrics
sc.pp.calculate_qc_metrics(adata_young, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)
sc.pp.calculate_qc_metrics(adata_old, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)

# Display the first few rows of the updated observation metadata
display(adata_young.obs.head())
display(adata_old.obs.head())

### Distribution of Mitochondrial Gene Percentage

High mitochondrial gene expression often indicates stressed or dying cells, which are typically considered low-quality. Visualizing the distribution of `pct_counts_mt` helps us determine a reasonable threshold to filter out these cells.

In [ ]:
# Violin plot for mitochondrial gene percentage in adata_young
fig, ax = plt.subplots(figsize=(6, 4))
sc.pl.violin(adata_young, ['pct_counts_mt'], jitter=0.4, show=False, ax=ax)
ax.set_title('Mitochondrial Gene Percentage for adata_young')
plt.tight_layout()
plt.show()

# Violin plot for mitochondrial gene percentage in adata_old
fig, ax = plt.subplots(figsize=(6, 4))
sc.pl.violin(adata_old, ['pct_counts_mt'], jitter=0.4, show=False, ax=ax)
ax.set_title('Mitochondrial Gene Percentage for adata_old')
plt.tight_layout()
plt.show()

### Filter Cells Based on QC Metrics

Based on the distributions observed, filter out cells that:
*   Have too few counts (low quality or empty droplets).
*   Have too many counts (doublets or large cells).
*   Express too few genes (low quality).
*   Have a high percentage of mitochondrial genes (stressed/dying cells).

In [ ]:
print('Filtering cells for adata_young...')
# Define filtering thresholds based on typical scRNA-seq QC and observed distributions
# These thresholds might need adjustment based on specific dataset characteristics
min_genes_young = 200
max_genes_young = 6000
min_counts_young = 500
max_counts_young = 20000
max_mito_pct_young = 10 # Example threshold, adjust based on violin plot

# filtering based on defined thresholds
sc.pp.filter_cells(adata_young, min_genes=min_genes_young)
sc.pp.filter_cells(adata_young, max_genes=max_genes_young)
sc.pp.filter_cells(adata_young, min_counts=min_counts_young)
sc.pp.filter_cells(adata_young, max_counts=max_counts_young)
adata_young = adata_young[adata_young.obs['pct_counts_mt'] < max_mito_pct_young, :]

print(f'Number of cells remaining in adata_young after filtering: {adata_young.n_obs}')

print('\nFiltering cells for adata_old...')
# Define filtering thresholds for adata_old
min_genes_old = 200
max_genes_old = 6000
min_counts_old = 500
max_counts_old = 20000
max_mito_pct_old = 10 # Example threshold, adjust based on violin plot

# filtering based on defined thresholds
sc.pp.filter_cells(adata_old, min_genes=min_genes_old)
sc.pp.filter_cells(adata_old, max_genes=max_genes_old)
sc.pp.filter_cells(adata_old, min_counts=min_counts_old)
sc.pp.filter_cells(adata_old, max_counts=max_counts_old)
adata_old = adata_old[adata_old.obs['pct_counts_mt'] < max_mito_pct_old, :]

print(f'Number of cells remaining in adata_old after filtering: {adata_old.n_obs}')

### Normalize and Log-Transform

* First normalize the total counts per cell to 1e4 (10,000) and then log-transform the data.

* Normalization helps to account for differences in sequencing depth between cells, and log-transformation helps to stabilize the variance and make the data more suitable for many statistical methods.

In [ ]:
print('Normalizing and log-transforming adata_young...')
sc.pp.normalize_total(adata_young, target_sum=1e4)
sc.pp.log1p(adata_young)
print('adata_young normalized and log-transformed.')

print('Normalizing and log-transforming adata_old...')
sc.pp.normalize_total(adata_old, target_sum=1e4)
sc.pp.log1p(adata_old)
print('adata_old normalized and log-transformed.')

---
# 03 Visualizations

## Distributions of Total Counts per Cell

In [ ]:
# Violin plot for total counts per cell in adata_young
fig, ax = plt.subplots(figsize=(6, 4))
sc.pl.violin(adata_young, ['total_counts'], jitter=0.4, show=False, ax=ax)
ax.set_title('Total Counts per Cell for adata_young')
plt.tight_layout()
plt.show()

# Violin plot for total counts per cell in adata_old
fig, ax = plt.subplots(figsize=(6, 4))
sc.pl.violin(adata_old, ['total_counts'], jitter=0.4, show=False, ax=ax)
ax.set_title('Total Counts per Cell for adata_old')
plt.tight_layout()
plt.show()

## Distribution of Number of Genes Expressed per Cell

In [ ]:
# Violin plot for number of genes by counts per cell in adata_young
fig, ax = plt.subplots(figsize=(6, 4))
sc.pl.violin(adata_young, ['n_genes_by_counts'], jitter=0.4, show=False, ax=ax)
ax.set_title('Number of Genes Expressed per Cell for adata_young')
plt.tight_layout()
plt.show()

# Violin plot for number of genes by counts per cell in adata_old
fig, ax = plt.subplots(figsize=(6, 4))
sc.pl.violin(adata_old, ['n_genes_by_counts'], jitter=0.4, show=False, ax=ax)
ax.set_title('Number of Genes Expressed per Cell for adata_old')
plt.tight_layout()
plt.show()

## Identify Highly Variable Genes

Identifying highly variable genes is a common step in single-cell RNA-seq analysis. These genes often represent the most biologically interesting features as they show significant variation across cells. Use `sc.pp.highly_variable_genes` to select these genes.

In [ ]:
# HVG for adata_young
print('Identifying highly variable genes for adata_young...')
sc.pp.highly_variable_genes(adata_young, min_mean=0.0125, max_mean=3, min_disp=0.5)
print(f'Number of highly variable genes in adata_young: {np.sum(adata_young.var.highly_variable)}')

# Display the top 5 highly variable genes
display(adata_young.var[adata_young.var.highly_variable].head())

# HVG for adata_old
print('Identifying highly variable genes for adata_old...')
sc.pp.highly_variable_genes(adata_old, min_mean=0.0125, max_mean=3, min_disp=0.5)
print(f'Number of highly variable genes in adata_old: {np.sum(adata_old.var.highly_variable)}')

# Display the top 5 highly variable genes
display(adata_old.var[adata_old.var.highly_variable].head())

## Plot Highly Variable Genes

Visualizes the mean expression versus dispersion for each gene, highlighting those identified as highly variable. This helps in understanding the characteristics of the selected genes.

In [ ]:
# HVG for adata_young
print('Plotting highly variable genes for adata_young...')
sc.pl.highly_variable_genes(adata_young, show=False)
plt.title('Highly Variable Genes for adata_young')
plt.tight_layout()
plt.show()

# HVG for adata_young
print('Plotting highly variable genes for adata_old...')
sc.pl.highly_variable_genes(adata_old, show=False)
plt.title('Highly Variable Genes for adata_old')
plt.tight_layout()
plt.show()

---
# 04 Dimensionality Reduction

## Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is used for dimensionality reduction. It transforms the data into a new set of orthogonal variables called principal components, which capture the most variance in the data. This helps in visualizing the data in a lower-dimensional space.

### PCA for `adata_young`

In [ ]:
# PCA for adata_young
print('Performing PCA for adata_young...')
sc.tl.pca(adata_young, svd_solver='arpack')
print('PCA for adata_young complete.')

print('Plotting PCA for adata_young...')
sc.pl.pca(adata_young, color='total_counts', show=False)
plt.title('PCA of adata_young (colored by total_counts)')
plt.tight_layout()
plt.show()

This plot visualizes the first two principal components, showing the main axes of variation in the dataset. Cells that are similar in their gene expression profile will cluster together.

### PCA for `adata_old`

Similarly, we will perform PCA for `adata_old` to reduce its dimensionality.

In [ ]:
# PCA for adata_old
print('Performing PCA for adata_old...')
sc.tl.pca(adata_old, svd_solver='arpack')
print('PCA for adata_old complete.')

print('Plotting PCA for adata_old...')
sc.pl.pca(adata_old, color='total_counts', show=False)
plt.title('PCA of adata_old (colored by total_counts)')
plt.tight_layout()
plt.show()

This plot visualizes the first two principal components for `adata_old`.

### Compute Neighbourhood Graphs

Before running UMAP, we need to compute a neighbourhood graph of cells. This graph connects cells that are transcriptionally similar, forming the basis for many downstream analyses, including clustering and visualization. We will use the PCA results to construct this graph.

In [ ]:
# Neighbourhood for adata_young
print('Computing neighbourhood graph for adata_young...')
sc.pp.neighbors(adata_young, n_neighbors=10, n_pcs=40)
print('Neighbourhood graph for adata_young computed.')

# Neighbourhood for adata_old
print('Computing neighbourhood graph for adata_old...')
sc.pp.neighbors(adata_old, n_neighbors=10, n_pcs=40)
print('Neighbourhood graph for adata_old computed.')

## Uniform Manifold Approximation and Projection (UMAP)

Uniform Manifold Approximation and Projection (UMAP) is a dimensionality reduction technique often used for visualizing single-cell data. It aims to place cells with similar gene expression profiles close to each other in a low-dimensional embedding, revealing the underlying structure of the data and will help to obtain a 2D representation of the cellular landscape.

In [ ]:
# UMAP for adata_young
print('Running UMAP for adata_young...')
sc.tl.umap(adata_young)
print('UMAP for adata_young complete.')

# UMAP for adata_old
print('Running UMAP for adata_old...')
sc.tl.umap(adata_old)
print('UMAP for adata_old complete.')

### Plot UMAPs

These plots visualize the UMAP embedding for `adata_young` and `adata_old`, showing the global structure and potential cell populations within the both cohorts.

In [ ]:
# UMAP plot for adata_young, colored by total_counts
print('Plotting UMAP for adata_young, colored by total_counts...')
sc.pl.umap(adata_young, color='total_counts', show=False)
plt.title('UMAP of adata_young (Colored by Total Counts)')
plt.tight_layout()
plt.show()

# UMAP plot for adata_old, colored by total_counts
print('Plotting UMAP for adata_old, colored by total_counts...')
sc.pl.umap(adata_old, color='total_counts', show=False)
plt.title('UMAP of adata_old (Colored by Total Counts)')
plt.tight_layout()
plt.show()

These plots visualize the high-dimensional single-cell RNA sequencing data in a 2D space, with each point representing a cell. The cells are colored by their 'total_counts', which represents the total number of reads detected for each cell. This visualization helps to assess if there are any batch effects or technical variations related to sequencing depth influencing the observed cellular structure. For example, if cells with very low or high total counts form distinct clusters, it might indicate technical rather than biological differences.

The plots show the global structure and potential cell populations within each cohort based on their gene expression profiles.

---
# 05 Clustering

Leiden clustering
* This algorithm identifies groups of transcriptionally similar cells, which are then assigned as clusters.

In [ ]:
# Ensure Leiden clustering has been performed on both adata_young and adata_old
# Clustering for Young
print('Performing Leiden clustering for adata_young...')
sc.tl.leiden(adata_young, key_added='leiden_clusters')
print('Leiden clustering for adata_young complete.')

# Clustering for Old
print('Performing Leiden clustering for adata_old...')
sc.tl.leiden(adata_old, key_added='leiden_clusters')
print('Leiden clustering for adata_old complete.')

print('Generating split UMAP plots colored by Leiden clusters...')
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# UMAP for adata_young
sc.pl.umap(adata_young, color='leiden_clusters', ax=axes[0], show=False, title='UMAP of Young Cohort (Leiden Clusters)', legend_loc='on data')

# UMAP for adata_old
sc.pl.umap(adata_old, color='leiden_clusters', ax=axes[1], show=False, title='UMAP of Old Cohort (Leiden Clusters)', legend_loc='on data')

plt.tight_layout()
plt.show()

The UMAP plots with Leiden clustering help to identify cell communities within each age cohort and visually represents them on a two-dimensional UMAP embedding, allowing for a comparison of cellular composition and structure between young and old glioblastoma microenvironments.

Leiden clusters, by themselves, are just numerical labels (e.g., '0', '1', '2'). They represent groups of cells that are transcriptionally similar according to the algorithm. To understand what they actually mean in a biological sense, we need to identify the genes that are uniquely or highly expressed within each cluster compared to other clusters. These are often referred to as *"marker genes"*.

Use `Differential Expression Analysis (DEA)` (also known as marker gene identification) to find these distinguishing genes for each cluster.

---
# 06 Differential Expression Analysis (DEA)

## Combine the cohorts for Differential Expression Analysis

To perform differential gene expression analysis between the young and old cohorts, first combine the two `AnnData` objects; `adata_young` and `adat_old`, into a single object. This allows us to compare gene expression profiles across both groups within a unified framework.

In [ ]:
print('Concatenating adata_young and adata_old...')
# Add a 'condition' column to each AnnData object
adata_young.obs['condition'] = 'young'
adata_old.obs['condition'] = 'old'

# Concatenate the AnnData objects
adata_combined = adata_young.concatenate(adata_old, join='inner', batch_key='condition')

print('Combined AnnData object created.')
print(adata_combined)

## Perform Differential Gene Expression Analysis

Now that the datasets are combined and labeled by condition, perform DEA to identify genes that are significantly upregulated or downregulated in the old cohort compared to the young cohort. We will use Scanpy's `rank_genes_groups` function, which can perform various statistical tests.

In [ ]:
print('Performing differential gene expression analysis...')
# Ensure the data is log-normalized before running rank_genes_groups if not already.
# If you have raw counts in adata_combined.X, you might want to re-normalize.
# For this analysis, assuming adata_combined.X contains log-normalized data after concatenation.

# Perform differential expression analysis using the 'condition' group
sc.tl.rank_genes_groups(adata_combined, 'condition', method='t-test')

print('Differential gene expression analysis complete.')

### Display Top Differentially Expressed Genes

Finally, display a table of the top differentially expressed genes between the `young` and `old` conditions. This will highlight genes that show significant differences in expression levels and could be indicative of biological changes associated with aging.

In [ ]:
print('Displaying top differentially expressed genes...')
# Get the differential expression results
result = adata_combined.uns['rank_genes_groups']

# Identify the actual group names from the structured array's dtype
group_field_names = result['names'].dtype.names # This will be ('0', '1') based on kernel state

if not group_field_names:
    print("No group fields found in the differential expression results.")
else:
    # Iterate through the identified group field names (e.g., '0', '1')
    for group_name in group_field_names:
        print(f"\n--- Top Differentially Expressed Genes for Group '{group_name}' ---")
        diff_genes = pd.DataFrame({
            'names': result['names'][group_name],
            'logfoldchanges': result['logfoldchanges'][group_name],
            'pvals_adj': result['pvals_adj'][group_name]
        })

        # Sort by logfoldchanges to see most upregulated/downregulated
        diff_genes_sorted = diff_genes.sort_values(by='logfoldchanges', ascending=False)

        # Display the top genes (e.g., top 10 most upregulated and top 10 most downregulated)
        print(f"Top 10 most upregulated genes in group '{group_name}':")
        display(diff_genes_sorted.head(10))
        print(f"Top 10 most downregulated genes in group '{group_name}':")
        display(diff_genes_sorted.tail(10))

print('\nTop differentially expressed genes displayed for all identified groups.')

### Visualize the top Differentially Expressed Genes on UMAP

Now visualize the expression levels of the top 10 most upregulated genes identified in each group:

* `Group 0` *(Young cohort)*
* `Group 1` *(Old cohort)*

This can provide insights into which cell populations are driving the differential expression of these genes.

In [ ]:
# Extract top 10 upregulated genes for Group '0' (Young)
print("Extracting top 10 upregulated genes for Group '0'...")
top_genes_group0 = adata_combined.uns['rank_genes_groups']['names']['0'][:10]

# Extract top 10 upregulated genes for Group '1' (Old)
print("Extracting top 10 upregulated genes for Group '1'...")
top_genes_group1 = adata_combined.uns['rank_genes_groups']['names']['1'][:10]

# Display the top 10 for each group
print(f"Top 10 upregulated genes in Group '0': {list(top_genes_group0)}")
print(f"Top 10 upregulated genes in Group '1': {list(top_genes_group1)}")

# Plot for the expression of these genes on the UMAP for Young cohort
print('Plotting expression of top genes on UMAP for Young...')
sc.pl.umap(adata_combined, color=top_genes_group0, cmap='viridis', s=10, ncols=4, show=False)
plt.suptitle(f"Top 10 Upregulated Genes in Group '0'(Young) on UMAP", y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

# Plot for the expression of these genes on the UMAP for Old cohort
print('Plotting expression of top genes on UMAP for Old...')
sc.pl.umap(adata_combined, color=top_genes_group1, cmap='viridis', s=10, ncols=4, show=False)
plt.suptitle(f"Top 10 Upregulated Genes in Group '1'(Old) on UMAP", y=1.02) # Add a main title
plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap
plt.show()

### Dotplot of Top Differentially Expressed Genes

To directly compare the expression patterns of the most upregulated genes in 'Group 0' and 'Group 1' (i.e., between the young and old cohorts), create a dotplot. This visualization shows both the average expression level *(color intensity)* and the proportion of cells expressing a gene *(dot size)* for each gene across the defined groups.

In [ ]:
print('Combining top genes from Group 0 and Group 1...')
# Get top 10 upregulated genes for Group '0' and Group '1'
top_genes_group0 = adata_combined.uns['rank_genes_groups']['names']['0'][:10].tolist()
top_genes_group1 = adata_combined.uns['rank_genes_groups']['names']['1'][:10].tolist()

# Combine and get unique genes
all_top_genes = list(set(top_genes_group0 + top_genes_group1))
print(f"Combined list of top genes: {all_top_genes}")

# Dotplot for these combined top genes across the 'condition' groups
print('Generating dotplot for combined top genes...')
sc.pl.dotplot(adata_combined, var_names=all_top_genes, groupby='condition', dendrogram=False, show=False)
plt.title('           Expression Across Conditions')
plt.tight_layout(rect=[1, 1, 1, 1])
plt.show()

### Violin Plot of Top Differentially Expressed Genes

To further visualize the distribution of expression for the top differentially expressed genes, generate violin plots. These plots show the probability density of the expression of each gene in both the `young` and `old` cohorts, providing more detail than the dotplot by illustrating the spread and peaks of the expression levels.

In [ ]:
print('Generating violin plots for combined top genes...')
# Create violin plots for the combined top genes across the 'condition' groups
# Using 'all_top_genes' which was defined in the previous cell

# Calculate the number of rows needed for the subplots
num_genes = len(all_top_genes)
ncols = 4 # Number of columns for the plots
nrows = (num_genes + ncols - 1) // ncols # Calculate rows needed

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 4, nrows * 3))
axes = axes.flatten() # Flatten the axes array for easy iteration

for i, gene in enumerate(all_top_genes):
    if i < len(axes): # Ensure we don't go out of bounds if there are more genes than subplots
        sc.pl.violin(adata_combined, keys=[gene], groupby='condition', jitter=0.4, ax=axes[i], show=False)
        axes[i].set_title(f'Expression of {gene}')
        axes[i].set_xlabel('') # Clear default x-label to avoid clutter
        axes[i].set_ylabel('Expression') # Set a generic y-label
        if i % ncols != 0: # Remove y-axis label for subsequent columns
            axes[i].set_ylabel('')
    else:
        break # Stop if all genes are plotted

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Violin Plots of Top Differentially Expressed Genes Across Conditions', y=1.02, fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent title overlap
plt.show()
print('Violin plots displayed.')

### Heatmap of Top Differentially Expressed Genes

To provide a comprehensive overview of the expression patterns of the top 20 differentially expressed genes, generate a heatmap. This visualization will show the relative expression levels of these genes across all samples, grouped by their *"condition"* `young vs. old`, allowing for clear identification of genes that are consistently upregulated or downregulated in one cohort compared to the other.

In [ ]:
print('Generating heatmap for combined top genes...')
# Create a heatmap for the combined top genes across the 'condition' groups
# Using 'all_top_genes' which was defined in the previous cell

sc.pl.heatmap(adata_combined, var_names=all_top_genes, groupby='condition', cmap='viridis', dendrogram=False, show=False)
plt.title('Heatmap of Top Differentially Expressed Genes Across Conditions')
plt.tight_layout()
plt.show()
print('Heatmap displayed.')

---
# 07 Modeling
## ML model for Age-Dependent Divergence

To address the question of how the age-dependent tumor microenvironment diverges, employ a machine learning approach:

1. Train a classification model
    * To distinguish between *"young"* and *"old"* cells based on their gene expression profiles.
2. Evaluate the model
    * Inspect the accuracy of our model
3. Analyze the feature importances
    * To identify the specific genes that contribute most significantly to the classification
    * Highlight key markers of divergence between the age groups.

In [ ]:
print('Preparing data for Machine Learning model...')

# 1. Select Features: Use the union of highly variable genes from both cohorts
# 'highly_variable-0' refers to adata_young, 'highly_variable-1' refers to adata_old
combined_hvgs_mask = adata_combined.var['highly_variable-0'] | adata_combined.var['highly_variable-1']
hvg_genes = adata_combined.var_names[combined_hvgs_mask].tolist()

# Ensure that all selected HVGs are present in adata_combined.var_names
hvg_genes_present = [gene for gene in hvg_genes if gene in adata_combined.var_names]

# Extract the expression matrix (X) for the selected HVGs
X = adata_combined[:, hvg_genes_present].X

# Convert sparse matrix to dense if necessary for scikit-learn models
if hasattr(X, 'toarray'):
    X = X.toarray()

# 2. Prepare Target Variable (y)
y = adata_combined.obs['condition']

# Encode the categorical target variable ('young', 'old') to numerical labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Get the class names for later interpretation
class_names = label_encoder.classes_
print(f"Encoded classes: {list(class_names)}")

# 3. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Data preparation complete. Using {len(hvg_genes_present)} highly variable genes as features.")

### Train Random Forest Classifier

Train a Random Forest Classifier, which is an ensemble learning method that builds multiple decision trees and merges them to get a more accurate and stable prediction. Random Forests are well-suited for high-dimensional data and can provide insights into feature importance.

In [ ]:
print('Training Random Forest Classifier...')

# Initialize and train the Random Forest Classifier
# Using n_estimators=100 for a balance of performance and computational cost
# random_state for reproducibility
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) # n_jobs=-1 uses all available cores
model.fit(X_train, y_train)

print('Random Forest Classifier training complete.')

### Evaluate Model Performance

After training, it's essential to evaluate the model's performance on unseen data *(the test set)*. This helps us understand how well the model generalizes and if it has learned meaningful patterns to differentiate between `young` and `old` conditions.

In [ ]:
print('Evaluating model performance...')

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Display classification report for more detailed metrics (precision, recall, f1-score)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

print('Model evaluation complete.')

### Identify Top Genes Contributing to Divergence (Feature Importance)

To answer the question about age-dependent divergence, examine the feature importances from the trained Random Forest model. These values indicate how much each gene contributes to the model's ability to classify cells as `young` or `old`. Genes with higher importance are more influential in distinguishing between the two conditions.

In [ ]:
print('Identifying top genes by feature importance...')

# Get feature importances from the trained model
feature_importances = model.feature_importances_

# Create a Pandas Series for easier handling, mapping importances to gene names
importance_df = pd.DataFrame({
    'gene': hvg_genes_present,
    'importance': feature_importances
})

# Sort by importance in descending order
importance_df = importance_df.sort_values(by='importance', ascending=False)

# Display the top 20 most important genes
print("\nTop 20 Genes Contributing to Age-Dependent Divergence:")
display(importance_df.head(20))

# Optionally, display the least important genes
# print("\nBottom 20 Genes (Least Important):")
# display(importance_df.tail(20))

print('Top genes identified.')

---
# 08 Gene Set Enrichment Analysis (GSEA)

## Gene Set Enrichment Analysis (GSEA)
A computational method that determines whether a defined set of genes shows statistically
significant, concordant differences between two biological states, in this case, age-dependent tumor microenvironment divergences.

As further investigation into the biological implications of this workflow, one can perform GSEA to identify biological pathways or functions that are significantly enriched among the differentially expressed genes. Use the `gseapy` library for this purpose, utilizing pre-ranked gene lists based on their log-fold changes.

In [ ]:
print('Checking available gene set libraries in gseapy...')
# Get a list of available gene set libraries
available_gene_sets = gp.get_library_name()

# Filter for mouse-specific GO and KEGG sets, or similar relevant sets
mouse_go_kegg_sets = [
    gs for gs in available_gene_sets
    if ('mouse' in gs.lower() or 'mus_musculus' in gs.lower()) and
       ('go_bp' in gs.lower() or 'kegg' in gs.lower() or 'gobp' in gs.lower())
]

print(f'Available mouse-related GO/KEGG gene sets: {mouse_go_kegg_sets}')

### For GSEA exploration to aid in diagnostic purposes.

Based on the output, in this case `['KEGG_2019_Mouse']`, One could select the most appropriate gene sets for the next GSEA run. For example, `MSigDB_GO_BP_Mouse` or `MSigDB_KEGG_Mouse`. For now, use placeholder names assuming a typical structure. One may need to adjust these based on the actual `available_gene_sets` output. If no specific mouse sets are found, generic ones could be used, but it's better to be species-specific.

Placeholders for corrected gene set names should be based on common `gseapy` conventions. Update these based on the `mouse_go_kegg_sets` output. For example:`['GO_Biological_Process_2018', 'KEGG_2019_Mouse']` Try to use the most recent ones first, in this case, use those from 2021 first if available, then otherwise adjust if needed.

One could update `gene_sets_to_use` based on the most recent available if found. The user would then need to review the `available_gene_sets` printout. If the exact '2021' sets aren't there, pick the closest ones. Where applicable, try to maintain the original naming scheme.

In [ ]:
# GSEA for Young and Old cohorts
print('Performing GSEA for each group...')
# Assuming 'result' and 'group_field_names' are available from the previous cell's execution
# result = adata_combined.uns['rank_genes_groups']
# group_field_names = result['names'].dtype.names

# Updated gene sets to use based on available_gene_sets output
gene_sets_to_use = ['KEGG_2019_Mouse'] # Using the identified available mouse KEGG gene set

for group_name in group_field_names:
    print(f"\n--- Running GSEA for Group '{group_name}' ---")

    # Create a DataFrame for this group's DE results
    group_de_df = pd.DataFrame({
        'gene': result['names'][group_name],
        'lfc': result['logfoldchanges'][group_name],
        'padj': result['pvals_adj'][group_name]
    })

    # Create a ranked list (Series) for GSEA: gene_name as index, lfc as values
    # Sorting by LFC is crucial for prerank GSEA
    ranked_gene_list = group_de_df.set_index('gene')['lfc'].sort_values(ascending=False)

    # Run GSEA using prerank method
    try:
        enr = gp.prerank(rnk=ranked_gene_list,
                         gene_sets=gene_sets_to_use,
                         outdir=f'gseapy_prerank_output_{group_name}',
                         seed=42, # for reproducibility
                         min_size=5,
                         max_size=500,
                         permutation_num=1000,
                         no_plot=True, # Set to False if you want plots generated
                         threads=1) # Changed from 'processes' to 'threads'

        print(f"Top 10 enriched gene sets for Group '{group_name}':")
        display(enr.res2d.head(10))
    except Exception as e:
        print(f"Error running GSEA for Group '{group_name}': {e}")

print('\nGSEA complete for all identified groups.')

---
# Conclusion: Unraveling Age-Dependent Divergence in Glioblastoma Microenvironments

This comprehensive single-cell RNA-seq analysis successfully investigated the age-dependent divergence within the glioblastoma tumor microenvironment (TME) in young versus aged mice. By processing and comparing data from two distinct cohorts, we leveraged a multi-faceted approach to identify key molecular and cellular distinctions.

Our workflow began with meticulous data preparation, including normalization, log-transformation, and quality control, ensuring robust downstream analysis. Dimensionality reduction techniques like PCA and UMAP effectively visualized the inherent cellular heterogeneity within each age group. Differential Gene Expression (DGE) analysis between the young and old cohorts highlighted numerous genes with significantly altered expression, offering direct molecular indicators of age-related changes. While Gene Set Enrichment Analysis (GSEA) explored enriched biological pathways, the lack of statistically significant findings after FDR correction suggests the need for further investigation into specific cellular contexts or alternative gene set databases.

Crucially, the application of a Random Forest Classifier provided a data-driven approach to pinpoint the most influential genes contributing to the TME's divergence. Achieving an accuracy of approximately 81.59% in distinguishing between young and old cells, the model identified a panel of top 20 genes (e.g., Hbb-bs, AC149090.1, Cd74, Ccl5, Avp) whose expression patterns are most predictive of the age phenotype. These genes represent compelling candidates for further functional validation, potentially serving as novel biomarkers or therapeutic targets in age-associated glioblastoma.

In essence, this report not only delineates the transcriptional landscape differences but also provides a prioritized list of genes that fundamentally distinguish the glioblastoma microenvironment in aged hosts. This knowledge is invaluable for understanding the mechanisms underlying age-related tumor progression and for developing age-appropriate therapeutic strategies.